#### Two Types of GPT
1. GPT-2 (what my script uses) 
    - open-weights, free, ~500MB. 
    - Hugging Face downloads the actual weight files to your machine, and model(input_ids) runs the real forward pass — embeddings, causal attention, FFN, all of it — locally, on your own CPU/GPU. 
    - No API key, no internet needed after the download, no cost per call. That's why the script works the way it does: you're holding the actual model.
2. GPT-3.5 / GPT-4 / GPT-4o / GPT-5 (what people usually mean by "GPT" today) 
    - OpenAI never released these weights. The only way to use them is to call OpenAI's hosted API: you send a prompt over HTTPS, their servers run the forward pass on their hardware, and you get text back. 
    - You never touch the model itself.

So: for GPT-2 → no API, run it yourself. For GPT-3/4/4o → yes, API is the only option (short of OpenAI changing their release policy).

Underneath, it's the same architecture and the same loop either way — tokenize → causal self-attention → predict next token → append → repeat (your Step 10). The API just hides all of that behind a request/response, and exposes the same knobs your notes already cover — temperature, top_p — as request parameters instead of Python arguments.

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "gpt2"  # GPT-2 small: 12 layers, 768 hidden, 12 heads, ~117M params
MAX_NEW_TOKENS = 20
SEED = 42

torch.manual_seed(SEED)

In [2]:
# ---------------------------------------------------------------------------
# 1. Load tokenizer & model (decoder-only, causal LM head)
# ---------------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

prompt = "Bajaj Finance Fixed Deposit interest rate for senior citizens is"

In [3]:
# ---------------------------------------------------------------------------
# 2. Tokenize + build the causal mask explicitly
#    (GPT2 does this internally - this just makes it visible)
# ---------------------------------------------------------------------------
inputs = tokenizer(prompt, return_tensors="pt").to(device)
print("Tokens:", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

seq_len = inputs["input_ids"].shape[1]
# 0 = allowed (can attend), -inf = blocked (future token, killed by softmax)
causal_mask = torch.triu(torch.full((seq_len, seq_len), float("-inf")), diagonal=1)
print("\nCausal mask (rows = query position, cols = key position):")
print(causal_mask)

Tokens: ['B', 'aj', 'aj', 'ĠFinance', 'ĠFixed', 'ĠDeposit', 'Ġinterest', 'Ġrate', 'Ġfor', 'Ġsenior', 'Ġcitizens', 'Ġis']

Causal mask (rows = query position, cols = key position):
tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0

In [4]:
# ---------------------------------------------------------------------------
# 3. Manual autoregressive generation loop
#    one token out -> append -> feed whole sequence back in -> repeat
# ---------------------------------------------------------------------------
def generate_greedy_manual(prompt, max_new_tokens=MAX_NEW_TOKENS):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(input_ids).logits          # (1, seq_len, vocab_size)
        next_token_logits = logits[0, -1, :]           # only the LAST position matters
        next_token_id = torch.argmax(next_token_logits).view(1, 1)
        input_ids = torch.cat([input_ids, next_token_id], dim=1)  # append -> repeat
        if next_token_id.item() == tokenizer.eos_token_id:
            break
    return tokenizer.decode(input_ids[0], skip_special_tokens=True)


print("\n=== Manual greedy generation ===")
print(generate_greedy_manual(prompt))


=== Manual greedy generation ===
Bajaj Finance Fixed Deposit interest rate for senior citizens is Rs. 1,000 per annum.

The government has also announced that it will increase


In [5]:
# ---------------------------------------------------------------------------
# 4. Decoding strategies via model.generate() (does the same loop, faster/safer)
# ---------------------------------------------------------------------------
def generate(prompt, **gen_kwargs):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    output_ids = model.generate(
        input_ids,
        max_new_tokens=MAX_NEW_TOKENS,
        pad_token_id=tokenizer.eos_token_id,
        **gen_kwargs,
    )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


print("\n=== Greedy decoding (do_sample=False) ===")
print(generate(prompt, do_sample=False))

print("\n=== Top-k sampling (k=50) ===")
print(generate(prompt, do_sample=True, top_k=50, temperature=1.0))

print("\n=== Top-p / nucleus sampling (p=0.9) ===")
print(generate(prompt, do_sample=True, top_p=0.9, temperature=1.0))

print("\n=== Low temperature (0.3) - focused/factual ===")
print(generate(prompt, do_sample=True, temperature=0.3, top_k=50))

print("\n=== High temperature (1.5) - random/creative ===")
print(generate(prompt, do_sample=True, temperature=1.5, top_k=50))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



=== Greedy decoding (do_sample=False) ===
Bajaj Finance Fixed Deposit interest rate for senior citizens is Rs. 1,000 per annum.

The government has also announced that it will increase

=== Top-k sampling (k=50) ===
Bajaj Finance Fixed Deposit interest rate for senior citizens is 1.25 per cent.

A senior government official said the policy would be implemented soon and

=== Top-p / nucleus sampling (p=0.9) ===
Bajaj Finance Fixed Deposit interest rate for senior citizens is 6.2%

5 July 2014 – In accordance with the Federal Tax Office's guidance (

=== Low temperature (0.3) - focused/factual ===
Bajaj Finance Fixed Deposit interest rate for senior citizens is Rs. 50 per month, while the rate for the non-resident is Rs. 50 per month

=== High temperature (1.5) - random/creative ===
Bajaj Finance Fixed Deposit interest rate for senior citizens is 25%-28%, which is higher than the rate provided by the Indian government today. It means that


##### Advance Models

In [6]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()  # reads .env in the current working directory

# Note: your .env uses lowercase "anthropic_api_key" — env var lookups are
# case-sensitive, so this must match exactly what's in the file.
api_key = os.getenv("anthropic_api_key")

client = anthropic.Anthropic(api_key=api_key)

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=200,
    temperature=0.3,  # same parameter, same meaning, as before
    messages=[
        {"role": "user", "content": "Bajaj Finance FD interest rate for senior citizens is"}
    ],
)

print(response)

Message(id='msg_01TJTEpiggbDV1D4nrY17FCh', container=None, content=[TextBlock(citations=None, text='## Bajaj Finance FD Interest Rates for Senior Citizens\n\nBajaj Finance offers **additional interest rates** for senior citizens (aged 60 years and above) over and above the regular FD rates.\n\n### Key Highlights:\n- Senior citizens typically get an **additional 0.25% p.a.** over regular rates\n- Interest rates generally range from approximately **7.40% to 8.85% p.a.** (varying by tenure)\n\n---\n\n⚠️ **Important Disclaimer:**\n> The exact interest rates **change frequently** and my information may not reflect the **current rates**.\n\n### For the Most Accurate & Updated Rates:\n- 🌐 Visit **[bajajfinserv.in](https://www.bajajfinserv.in)**\n- 📞 Call Bajaj Finance customer care: **1800-103-3535', type='text')], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason='max_tokens', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_

In [7]:
print(response.content[0].text)

## Bajaj Finance FD Interest Rates for Senior Citizens

Bajaj Finance offers **additional interest rates** for senior citizens (aged 60 years and above) over and above the regular FD rates.

### Key Highlights:
- Senior citizens typically get an **additional 0.25% p.a.** over regular rates
- Interest rates generally range from approximately **7.40% to 8.85% p.a.** (varying by tenure)

---

⚠️ **Important Disclaimer:**
> The exact interest rates **change frequently** and my information may not reflect the **current rates**.

### For the Most Accurate & Updated Rates:
- 🌐 Visit **[bajajfinserv.in](https://www.bajajfinserv.in)**
- 📞 Call Bajaj Finance customer care: **1800-103-3535


In [8]:
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=200,   # was 20 — too small for any real answer
    temperature=0.1,  # lower temperature = more factual, less creative
    messages=[
        {"role": "user", "content": "Bajaj Finance FD interest rate for senior citizens is"}
    ],
)
print(response.content[0].text)

## Bajaj Finance FD Interest Rates for Senior Citizens

Bajaj Finance offers **additional interest rates** for senior citizens (aged 60 years and above) over and above the regular FD rates.

### Key Highlights:
- Senior citizens typically get an **additional 0.25% p.a.** over regular rates
- The rates generally range from approximately **7.4% to 8.85% p.a.** (varying by tenure)

---

⚠️ **Important Disclaimer:**
> My knowledge has a cutoff, and **FD interest rates change frequently**. The rates I may have are likely **outdated**.

### For the most accurate & current rates, please check:
- 🌐 **Official Website:** [www.bajajfinserv.in](https://www.bajajfinserv.in)
- 📞 **Customer


In [9]:
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=200,   # was 20 — too small for any real answer
    temperature=0.1,  # lower temperature = more factual, less creative
    messages=[
        {"role": "user", "content": "Capital of India is"}
    ],
)
print(response.content[0].text)

The capital of India is **New Delhi**. 🇮🇳

It has been the capital since 1911, when the British colonial government moved the capital from Calcutta (now Kolkata) to Delhi. New Delhi is located in the National Capital Territory (NCT) of Delhi and serves as the seat of the Indian government.


In [10]:
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=10,   # was 20 — too small for any real answer
    temperature=0.1,  # lower temperature = more factual, less creative
    messages=[
        {"role": "user", "content": "Capital of India is"}
    ],
)
print(response.content[0].text)

The capital of India is **New Delhi**.
